# Pooling Generation (Qrels)
This notebook generates the evaluation pool by extracting the top results from BM25, Dense, and Hybrid search for a set of test queries.

In [1]:
import pandas as pd
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..')))
from src.search_engine import BM25Searcher, DenseSearcher, HybridSearcher

# 50 TEST QUERIES CATEGORIZED 
TEST_QUERIES = [
    # Tramas, conceptos y emociones sin nombres propios
    "A story of survival in extreme cold weather and snow",
    "Psychological tension and family secrets between relatives",
    "High school students forming a music band and dealing with youth problems",
    "A journey through outer space to find a new habitable planet for humanity",
    "Documentary showing the history of jazz music and its cultural impact",
    "A detective dealing with his own inner demons while chasing a serial killer",
    "Romantic relationship between people from completely different social classes",
    "A struggle against an oppressive dystopian government in a near future",
    "Feel-good comedy about an unexpected friendship between a senior and a teenager",
    "Athletes overcoming severe physical injuries to win a championship",
    "A house haunted by malevolent spirits that terrorize a family",
    "Time travel causing complex paradoxes and altering historical events",
    "An deep dive into the lives of wild animals in the African savannah",
    "A lawyer fighting against a corrupt corporation to protect poor citizens",
    "An artist struggling with writer's block and losing their mind",
    "A classic revenge story where a protagonist hunts down former betrayers",
    "Coming of age story of a young boy finding his identity in a small town",

    # Directores, actores, combinaciones
    "Movies directed by Steven Spielberg",
    "Movies directed by Martin Scorsese",
    "Films starring Samuel L. Jackson",
    "Movies starring Danny Trejo",
    "Films starring Shah Rukh Khan",
    "Movies starring Amitabh Bachchan",
    "Films starring Nicolas Cage",
    "Movies starring Bruce Willis",
    "Movies directed by Robert Vince",
    "Films directed by Paul Hoen",
    "Movies starring Anupam Kher",
    "Films starring Maggie Binkley",
    "Movies starring Paresh Rawal",
    "Films starring Naseeruddin Shah",
    "Movies starring Akshay Kumar",
    "Films directed by John Lasseter",
    "Movies starring Om Puri",

    # Entidad + Concepto
    "Action movies starring Samuel L. Jackson",
    "Science fiction movies directed by Steven Spielberg",
    "Animated films directed by John Lasseter",
    "Documentaries featuring Elon Musk",
    "Drama movies starring Nicolas Cage",
    "Romantic comedy dramas starring Shah Rukh Khan",
    "Horror movies starring Danny Trejo",
    "Crime drama thriller directed by Martin Scorsese",
    "Comedy movies starring Bruce Willis",
    "Action thrillers starring Amitabh Bachchan",
    "Sports documentary starring or about athletes",
    "Family comedy directed by Robert Vince",
    "Musical comedy directed by Paul Hoen",
    "Comedy movies starring Anupam Kher",
    "Drama movies starring Naseeruddin Shah",
    "Action movies starring Akshay Kumar",

    "A science fiction movie about space exploration starring Matthew McConaughey",
    "A gritty crime thriller about a serial killer starring Anthony Hopkins",
    "Romantic comedy set in New York directed by Nora Ephron",
    "Action movie involving a bank robbery or hostage situation starring Keanu Reeves",
    "A coming of age drama in a small town starring Timothée Chalamet",
    "A fantasy movie about magic, elves or wizards directed by Peter Jackson",
    "Documentary about nature, animals and the ocean narrated by David Attenborough",
    "A dramatic war movie set during World War II directed by Quentin Tarantino",
    "A feel-good family movie about dogs or pets starring Paul Walker",
    "A suspenseful mystery or psychological thriller starring Leonardo DiCaprio"
]

In [2]:
# Load the dataframe with embeddings
print("Loading dataframe...")
movies = pd.read_pickle('../data/movies_with_embeddings.pkl')

# Initialize searchers
bm25_searcher = BM25Searcher(movies)
dense_searcher = DenseSearcher(movies)
hybrid_searcher = HybridSearcher(bm25_searcher, dense_searcher)


Loading dataframe...
Tokenizing corpus for BM25...


Building BM25 Index...
Saving BM25 index to /teamspace/studios/this_studio/semantic-movie-retrieval-nlp/data/bm25_index.pkl...


BM25 Index built and saved successfully.
Loading SentenceTransformer model (multi-qa-MiniLM-L6-cos-v1)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Building FAISS Index...
Saving FAISS index to /teamspace/studios/this_studio/semantic-movie-retrieval-nlp/data/faiss_index.bin...
FAISS Index built and saved successfully.


In [3]:
# Generate Pool
pool_rows = []
top_k = 10
global_id = 0

for query in TEST_QUERIES:
    print(f"Processing query: '{query}'")
    bm25_res = bm25_searcher.search(query, top_k=top_k)
    dense_res = dense_searcher.search(query, top_k=top_k)
    hybrid_res = hybrid_searcher.search(query, top_k=top_k)
    
    unique_titles = set()
    for res in bm25_res + dense_res + hybrid_res:
        unique_titles.add(res['title'])
        
    for title in sorted(unique_titles):
        pool_rows.append({
            'id': global_id,
            'query': query,
            'movie_title': title,
            'relevance': ''
        })
        global_id += 1

pool_df = pd.DataFrame(pool_rows)
print(f"\nGenerated pool with {len(pool_df)} unique query-movie pairs.")

# Formatter for copy-pasting to the LLM 
movie_desc_dict = dict(zip(movies['title'], movies['content_narrative']))

prompt_content = []
prompt_header = """Instructions for the AI Labeler:
You are an expert movie relevance annotator.
For each item below, you are given a unique ID, a Search Query, a Movie Title, and a Narrative Content (synopsis and details).
Grade the relevance of the movie to the query on a scale of 0 to 5:
0: Completely irrelevant (no connection whatsoever)
1: Barely relevant (very weak or accidental keyword overlap)
2: Somewhat relevant (shares some elements like genre or minor themes, but not matching core intent)
3: Relevant (matches the main concepts/entities of the search query, but might miss specific aspects)
4: Highly relevant (excellent match to the query, covers the search intent very well)
5: Perfect match (exact match to the query, or is the precise movie/franchise/subject requested)

Return your response as a simple list where each line has the format:
ID: Score
Example:
0: 5
1: 0
2: 3

Here are the items to grade:
"""
prompt_content.append(prompt_header)

for _, row in pool_df.iterrows():
    row_id = row['id']
    q = row['query']
    title = row['movie_title']
    desc = movie_desc_dict.get(title, "No description available.")
    
    item_str = f"ID: {row_id}\nQuery: {q}\nMovie Title: {title}\nNarrative Content: {desc}\n--------------------------------------------------\n"
    prompt_content.append(item_str)

output_prompt_path = '../data/prompt_for_ai.txt'
with open(output_prompt_path, 'w', encoding='utf-8') as f:
    f.writelines(prompt_content)

print(f"AI Prompt successfully exported to '{output_prompt_path}'.")
print("Preview of the first 3 items:")
print("".join(prompt_content[1:4]))

Processing query: 'A story of survival in extreme cold weather and snow'


Processing query: 'Psychological tension and family secrets between relatives'
Processing query: 'High school students forming a music band and dealing with youth problems'
Processing query: 'A journey through outer space to find a new habitable planet for humanity'
Processing query: 'Documentary showing the history of jazz music and its cultural impact'
Processing query: 'A detective dealing with his own inner demons while chasing a serial killer'
Processing query: 'Romantic relationship between people from completely different social classes'
Processing query: 'A struggle against an oppressive dystopian government in a near future'
Processing query: 'Feel-good comedy about an unexpected friendship between a senior and a teenager'
Processing query: 'Athletes overcoming severe physical injuries to win a championship'
Processing query: 'A house haunted by malevolent spirits that terrorize a family'
Processing query: 'Time travel causing complex paradoxes and altering historical events'


In [8]:
# Results Importer
import re
import os

ai_output_raw = """
0: 4
1: 0
2: 5
3: 1
4: 1
5: 1
6: 2
7: 1
8: 2
9: 5
10: 4
11: 0
12: 3
13: 0
14: 2
15: 2
16: 1
17: 2
18: 2
19: 4
20: 3
21: 1
22: 1
23: 2
24: 3
25: 3
26: 4
27: 4
28: 1
29: 0
30: 2
31: 2
32: 0
33: 2
34: 1
35: 0
36: 2
37: 3
38: 2
39: 4
40: 5
41: 2
42: 1
43: 2
44: 1
45: 0
46: 2
47: 5
48: 1
49: 0
50: 4
51: 4
52: 3
53: 3
54: 1
55: 0
56: 1
57: 1
58: 1
59: 0
60: 1
61: 0
62: 0
63: 1
64: 1
65: 1
66: 1
67: 0
68: 1
69: 4
70: 2
71: 2
72: 1
73: 1
74: 0
75: 1
76: 1
77: 2
78: 2
79: 1
80: 3
81: 2
82: 0
83: 0
84: 1
85: 3
86: 4
87: 0
88: 1
89: 2
90: 0
91: 1
92: 0
93: 2
94: 5
95: 0
96: 2
97: 2
98: 0
99: 1
100: 0
101: 1
102: 4
103: 1
104: 2
105: 0
106: 5
107: 4
108: 1
109: 4
110: 2
111: 3
112: 5
113: 1
114: 4
115: 4
116: 1
117: 4
118: 0
119: 2
120: 0
121: 1
122: 5
123: 0
124: 5
125: 5
126: 1
127: 1
128: 2
129: 3
130: 1
131: 2
132: 1
133: 1
134: 4
135: 5
136: 5
137: 0
138: 2
139: 0
140: 5
141: 0
142: 0
143: 0
144: 5
145: 3
146: 0
147: 0
148: 0
149: 0
150: 1
151: 1
152: 4
153: 0
154: 3
155: 1
156: 4
157: 0
158: 0
159: 4
160: 2
161: 0
162: 0
163: 2
164: 1
165: 1
166: 2
167: 2
168: 2
169: 1
170: 1
171: 5
172: 2
173: 3
174: 0
175: 0
176: 2
177: 0
178: 0
179: 0
180: 5
181: 1
182: 1
183: 1
184: 1
185: 0
186: 0
187: 0
188: 1
189: 0
190: 3
191: 0
192: 3
193: 0
194: 0
195: 0
196: 0
197: 0
198: 0
199: 0
200: 2
201: 5
202: 5
203: 1
204: 1
205: 1
206: 1
207: 1
208: 1
209: 0
210: 1
211: 2
212: 1
213: 3
214: 0
215: 0
216: 1
217: 1
218: 0
219: 1
220: 4
221: 1
222: 1
223: 1
224: 3
225: 3
226: 3
227: 4
228: 2
229: 2
230: 1
231: 2
232: 3
233: 1
234: 2
235: 3
236: 4
237: 3
238: 3
239: 3
240: 0
241: 3
242: 4
243: 3
244: 5
245: 4
246: 5
247: 3
248: 0
249: 0
250: 0
251: 2
252: 2
253: 1
254: 0
255: 0
256: 5
257: 0
258: 5
259: 4
260: 3
261: 2
262: 3
263: 4
264: 3
265: 0
266: 0
267: 1
268: 2
269: 3
270: 3
271: 0
272: 1
273: 5
274: 0
275: 5
276: 3
277: 2
278: 0
279: 0
280: 5
281: 0
282: 2
283: 2
284: 0
285: 3
286: 1
287: 1
288: 2
289: 2
290: 1
291: 0
292: 3
293: 1
294: 1
295: 3
296: 3
297: 0
298: 1
299: 1
300: 2
301: 4
302: 1
303: 1
304: 1
305: 5
306: 0
307: 0
308: 1
309: 1
310: 5
311: 1
312: 2
313: 0
314: 3
315: 4
316: 1
317: 1
318: 0
319: 3
320: 3
321: 2
322: 4
323: 3
324: 2
325: 0
326: 1
327: 2
328: 3
329: 1
330: 1
331: 0
332: 1
333: 0
334: 2
335: 2
336: 1
337: 1
338: 5
339: 0
340: 0
341: 0
342: 3
343: 1
344: 2
345: 2
346: 2
347: 3
348: 4
349: 1
350: 3
351: 3
352: 2
353: 3
354: 2
355: 1
356: 3
357: 2
358: 4
359: 3
360: 3
361: 1
362: 0
363: 1
364: 3
365: 4
366: 3
367: 1
368: 1
369: 1
370: 3
371: 4
372: 2
373: 4
374: 4
375: 1
376: 1
377: 2
378: 3
379: 1
380: 1
381: 1
382: 2
383: 1
384: 2
385: 2
386: 2
387: 2
388: 1
389: 0
390: 0
391: 0
392: 5
393: 0
394: 5
395: 5
396: 5
397: 5
398: 5
399: 0
400: 5
401: 5
402: 5
403: 5
404: 0
405: 0
406: 0
407: 5
408: 0
409: 5
410: 0
411: 0
412: 1
413: 0
414: 5
415: 0
416: 0
417: 0
418: 5
419: 5
420: 0
421: 0
422: 0
423: 5
424: 0
425: 5
426: 1
427: 0
428: 5
429: 5
430: 5
431: 0
432: 5
433: 0
434: 5
435: 0
436: 0
437: 0
438: 0
439: 5
440: 5
441: 0
442: 0
443: 0
444: 5
445: 5
446: 0
447: 0
448: 0
449: 5
450: 0
451: 0
452: 0
453: 5
454: 5
455: 5
456: 5
457: 5
458: 5
459: 5
460: 5
461: 0
462: 0
463: 0
464: 5
465: 5
466: 5
467: 5
468: 0
469: 0
470: 0
471: 0
472: 5
473: 5
474: 5
475: 0
476: 0
477: 0
478: 5
479: 5
480: 5
481: 5
482: 5
483: 5
484: 5
485: 5
486: 0
487: 5
488: 5
489: 0
490: 5
491: 0
492: 0
493: 5
494: 5
495: 0
496: 5
497: 0
498: 0
499: 5
500: 0
501: 5
502: 0
503: 0
504: 5
505: 0
506: 0
507: 0
508: 0
509: 5
510: 5
511: 5
512: 0
513: 5
514: 5
515: 0
516: 5
517: 5
518: 0
519: 5
520: 5
521: 5
522: 5
523: 0
524: 0
525: 5
526: 5
527: 5
528: 5
529: 5
530: 0
531: 5
532: 0
533: 5
534: 0
535: 5
536: 5
537: 0
538: 0
539: 0
540: 5
541: 5
542: 5
543: 5
544: 5
545: 0
546: 0
547: 0
548: 0
549: 0
550: 5
551: 5
552: 5
553: 0
554: 0
555: 5
556: 5
557: 5
558: 5
559: 5
560: 5
561: 0
562: 0
563: 0
564: 0
565: 5
566: 0
567: 0
568: 0
569: 0
570: 0
571: 0
572: 5
573: 5
574: 5
575: 5
576: 0
577: 0
578: 0
579: 5
580: 5
581: 0
582: 0
583: 0
584: 0
585: 5
586: 0
587: 5
588: 0
589: 5
590: 0
591: 0
592: 0
593: 0
594: 0
595: 5
596: 0
597: 0
598: 0
599: 5
600: 0
601: 0
602: 0
603: 5
604: 0
605: 5
606: 5
607: 0
608: 5
609: 5
610: 5
611: 0
612: 5
613: 5
614: 5
615: 5
616: 0
617: 5
618: 5
619: 5
620: 0
621: 5
622: 0
623: 0
624: 0
625: 0
626: 0
627: 5
628: 5
629: 5
630: 5
631: 0
632: 5
633: 5
634: 0
635: 5
636: 5
637: 5
638: 5
639: 5
640: 5
641: 5
642: 5
643: 5
644: 5
645: 5
646: 5
647: 5
648: 0
649: 0
650: 0
651: 0
652: 0
653: 0
654: 0
655: 0
656: 0
657: 0
658: 0
659: 0
660: 0
661: 5
662: 5
663: 5
664: 5
665: 0
666: 5
667: 5
668: 0
669: 0
670: 0
671: 0
672: 0
673: 0
674: 0
675: 5
676: 5
677: 0
678: 5
679: 5
680: 0
681: 5
682: 0
683: 0
684: 0
685: 5
686: 5
687: 5
688: 5
689: 5
690: 0
691: 0
692: 5
693: 5
694: 0
695: 0
696: 5
697: 0
698: 0
699: 0
700: 5
701: 0
702: 0
703: 5
704: 5
705: 5
706: 5
707: 0
708: 0
709: 0
710: 0
711: 5
712: 5
713: 0
714: 5
715: 0
716: 0
717: 0
718: 0
719: 0
720: 0
721: 0
722: 0
723: 0
724: 5
725: 5
726: 5
727: 5
728: 5
729: 5
730: 0
731: 0
732: 5
733: 5
734: 5
735: 5
736: 5
737: 0
738: 0
739: 0
740: 0
741: 5
742: 5
743: 0
744: 0
745: 5
746: 0
747: 0
748: 0
749: 1
750: 5
751: 5
752: 0
753: 5
754: 5
755: 5
756: 5
757: 5
758: 5
759: 0
760: 5
761: 0
762: 5
763: 0
764: 0
765: 0
766: 0
767: 5
768: 5
769: 5
770: 0
771: 0
772: 0
773: 5
774: 0
775: 0
776: 0
777: 0
778: 0
779: 5
780: 5
781: 2
782: 0
783: 0
784: 0
785: 0
786: 5
787: 2
788: 0
789: 5
790: 0
791: 0
792: 0
793: 5
794: 0
795: 5
796: 5
797: 5
798: 0
799: 0
800: 1
801: 0
802: 0
803: 0
804: 5
805: 3
806: 3
807: 1
808: 0
809: 0
810: 0
811: 0
812: 5
813: 0
814: 0
815: 0
816: 0
817: 1
818: 0
819: 0
820: 0
821: 0
822: 5
823: 0
824: 0
825: 5
826: 5
827: 5
828: 5
829: 5
830: 5
831: 0
832: 5
833: 0
834: 5
835: 0
836: 0
837: 1
838: 0
839: 5
840: 5
841: 5
842: 0
843: 0
844: 0
845: 0
846: 0
847: 0
848: 0
849: 0
850: 5
851: 0
852: 5
853: 0
854: 5
855: 0
856: 0
857: 0
858: 0
859: 0
860: 0
861: 0
862: 0
863: 0
864: 0
865: 2
866: 3
867: 0
868: 5
869: 0
870: 0
871: 3
872: 2
873: 2
874: 2
875: 2
876: 2
877: 5
878: 0
879: 5
880: 0
881: 0
882: 5
883: 0
884: 0
885: 0
886: 0
887: 0
888: 0
889: 5
890: 5
891: 4
892: 5
893: 2
894: 2
895: 5
896: 5
897: 5
898: 0
899: 5
900: 0
901: 4
902: 0
903: 1
904: 0
905: 4
906: 5
907: 0
908: 0
909: 0
910: 2
911: 1
912: 1
913: 1
914: 1
915: 0
916: 0
917: 5
918: 2
919: 0
920: 1
921: 0
922: 0
923: 0
924: 2
925: 5
926: 0
927: 0
928: 0
929: 0
930: 0
931: 4
932: 0
933: 0
934: 0
935: 0
936: 5
937: 5
938: 0
939: 0
940: 0
941: 0
942: 0
943: 0
944: 1
945: 0
946: 0
947: 5
948: 0
949: 5
950: 1
951: 0
952: 0
953: 3
954: 2
955: 2
956: 1
957: 2
958: 2
959: 0
960: 0
961: 0
962: 0
963: 2
964: 2
965: 5
966: 0
967: 0
968: 0
969: 0
970: 0
971: 1
972: 1
973: 1
974: 5
975: 0
976: 1
977: 0
978: 0
979: 1
980: 0
981: 0
982: 0
983: 5
984: 0
985: 0
986: 0
987: 0
988: 4
989: 4
990: 2
991: 0
992: 4
993: 0
994: 0
995: 5
996: 0
997: 4
998: 3
999: 0
1000: 4
1001: 4
1002: 5
1003: 5
1004: 5
1005: 5
1006: 3
1007: 3
1008: 4
1009: 5
1010: 5
1011: 5
1012: 5
1013: 5
1014: 5
1015: 5
1016: 3
1017: 2
1018: 0
1019: 2
1020: 5
1021: 2
1022: 4
1023: 5
1024: 0
1025: 0
1026: 0
1027: 0
1028: 0
1029: 0
1030: 0
1031: 5
1032: 5
1033: 0
1034: 5
1035: 0
1036: 5
1037: 5
1038: 5
1039: 0
1040: 5
1041: 0
1042: 0
1043: 0
1044: 3
1045: 5
1046: 0
1047: 0
1048: 5
1049: 2
1050: 0
1051: 5
1052: 2
1053: 0
1054: 0
1055: 0
1056: 0
1057: 0
1058: 2
1059: 0
1060: 2
1061: 0
1062: 2
1063: 0
1064: 0
1065: 0
1066: 0
1067: 0
1068: 2
1069: 0
1070: 5
1071: 0
1072: 0
1073: 5
1074: 5
1075: 5
1076: 2
1077: 3
1078: 1
1079: 0
1080: 0
1081: 2
1082: 5
1083: 5
1084: 0
1085: 0
1086: 2
1087: 5
1088: 2
1089: 5
1090: 0
1091: 0
1092: 3
1093: 2
1094: 0
1095: 4
1096: 5
1097: 0
1098: 0
1099: 3
1100: 5
1101: 0
1102: 5
1103: 0
1104: 5
1105: 5
1106: 3
1107: 0
1108: 4
1109: 4
1110: 5
1111: 5
1112: 5
1113: 0
1114: 4
1115: 4
1116: 0
1117: 0
1118: 0
1119: 0
1120: 0
1121: 0
1122: 5
1123: 1
1124: 0
1125: 1
1126: 5
1127: 5
1128: 5
1129: 1
1130: 0
1131: 0
1132: 1
1133: 1
1134: 1
1135: 1
1136: 1
1137: 0
1138: 1
1139: 1
1140: 1
1141: 0
1142: 0
1143: 2
1144: 1
1145: 1
1146: 1
1147: 1
1148: 1
1149: 1
1150: 1
1151: 1
1152: 2
1153: 2
1154: 2
1155: 0
1156: 2
1157: 1
1158: 2
1159: 2
1160: 2
1161: 2
1162: 1
1163: 1
1164: 1
1165: 2
1166: 1
1167: 1
1168: 2
1169: 1
1170: 2
1171: 3
1172: 1
1173: 1
1174: 2
1175: 3
1176: 2
1177: 3
1178: 1
1179: 2
1180: 2
1181: 3
1182: 3
1183: 5
1184: 2
1185: 2
1186: 0
1187: 1
1188: 0
1189: 1
1190: 0
1191: 0
1192: 2
1193: 1
1194: 0
1195: 1
1196: 2
1197: 0
1198: 2
1199: 3
1200: 1
1201: 4
1202: 2
1203: 1
1204: 2
1205: 1
1206: 2
1207: 1
1208: 2
1209: 1
1210: 3
1211: 2
1212: 3
1213: 2
1214: 2
1215: 2
1216: 2
1217: 2
1218: 1
1219: 3
1220: 0
1221: 0
1222: 1
1223: 0
1224: 1
1225: 1
1226: 2
1227: 2
1228: 2
1229: 1
1230: 4
1231: 1
1232: 0
1233: 4
1234: 1
1235: 1
1236: 2
1237: 1
1238: 3
1239: 2
1240: 2
1241: 0
1242: 1
1243: 0
1244: 1
1245: 2
1246: 1
1247: 2
1248: 2
1249: 0
1250: 1
1251: 2
1252: 0
1253: 2
1254: 1
1255: 2
1256: 2
1257: 1
1258: 2
1259: 5
1260: 5
1261: 1
1262: 1
1263: 2
1264: 1
1265: 2
1266: 2
1267: 5
1268: 5
1269: 4
1270: 3
1271: 5
1272: 5
1273: 5
1274: 2
1275: 4
1276: 4
1277: 5
1278: 3
1279: 2
1280: 2
1281: 2
1282: 5
1283: 3
1284: 2
1285: 2
1286: 2
1287: 1
1288: 1
1289: 2
1290: 2
1291: 2
1292: 3
1293: 2
1294: 0
1295: 0
1296: 3
1297: 2
1298: 2
1299: 2
1300: 2
1301: 0
1302: 2
1303: 2
1304: 0
1305: 2
1306: 2
1307: 3
1308: 3
1309: 5
1310: 0
1311: 3
1312: 0
1313: 1
1314: 2
1315: 0
1316: 0
1317: 3
1318: 3
1319: 3
1320: 3
1321: 3
1322: 0
1323: 3
1324: 1
1325: 3
1326: 3
1327: 1
1328: 0
1329: 3
1330: 2
1331: 3
1332: 1
1333: 3
1334: 3
1335: 1
1336: 5
1337: 2
1338: 2
1339: 3
1340: 3
1341: 3
1342: 3
1343: 5
1344: 3
1345: 3
1346: 4
1347: 3
1348: 2
1349: 2
1350: 2
1351: 2
1352: 3
1353: 2
"""

# Parse grades using regular expression
grade_pattern = re.compile(r'(?:ID:?\s*)?(\d+)\s*[\s,:-]+\s*([0-5])', re.IGNORECASE)
parsed_grades = {}

for line in ai_output_raw.strip().split('\n'):
    line = line.strip()
    if not line or line.startswith('#'):
        continue
    match = grade_pattern.search(line)
    if match:
        item_id = int(match.group(1))
        score = int(match.group(2))
        parsed_grades[item_id] = score

print(f"Parsed {len(parsed_grades)} grades from the AI output.")

if len(parsed_grades) > 0:
    pool_df['relevance'] = pool_df['id'].map(parsed_grades)
    missing_count = pool_df['relevance'].isna().sum()
    if missing_count > 0:
        print(f"Warning: {missing_count} rows were not graded. Setting their relevance to 0.")
        pool_df['relevance'] = pool_df['relevance'].fillna(0)
    else:
        print("All rows successfully graded!")
        
    pool_df['relevance'] = pool_df['relevance'].astype(int)
    output_df = pool_df[['query', 'movie_title', 'relevance']].copy()
    
    output_csv = '../data/qrels_to_grade.csv'
    output_df.to_csv(output_csv, index=False)
    print(f"Successfully exported final graded qrels to '{output_csv}' with {len(output_df)} rows.")
else:
    print("No grades were parsed. Please paste the AI output in 'ai_output_raw' or write to 'ai_grades.txt' and rerun this cell.")

Parsed 1354 grades from the AI output.
All rows successfully graded!
Successfully exported final graded qrels to 'qrels_to_grade.csv' with 1354 rows.
